# ARC-AGI-3 Component-Skill Prefill + Scored Run Notebook

This notebook is built for Kaggle competition reruns where internet is disabled and all useful material must already be attached under `/kaggle/input`.

It performs a **full input discovery pass** before play:

1. Inventory every attached file/folder under `/kaggle/input`.
2. Classify notebooks, Python utilities, packages/wheels/zips, JSON/JSONL/CSV/Parquet datasets, text docs, and prior replay/skill files.
3. Statically extract candidate solver components from notebooks/scripts/utilities without executing public code.
4. Ingest compatible skill libraries and transition/replay datasets when present.
5. Rank the strongest detected components and convert them into `data/skills/skill_library.json`.
6. Export `agent/my_agent.py` with the graph-memory/object-targeting policy.
7. Optionally run an official ARC scorecard through `arc_agi`.

No online scraping is used. Add public notebooks, datasets, and utility folders as Kaggle input datasets.


## 0. Runtime setup

Environment variables you can set in Kaggle:

```bash
NINE_ARC_RUN_SCORECARD=1
NINE_ARC_GAMES=all
NINE_ARC_MAX_STEPS=240
NINE_ARC_EPSILON=0.055
NINE_ARC_SCAN_MAX_MB=16
NINE_ARC_IMPORT_UTIL_PATHS=1
```


In [ ]:
from pathlib import Path
import os, sys, json, time, random, hashlib, re, ast, csv, zipfile, traceback, math
from dataclasses import dataclass, asdict, field
from collections import Counter, defaultdict
from typing import Any, Dict, List, Optional, Tuple, Iterable, Set

SEED = int(os.environ.get("NINE_ARC_SEED", "918"))
random.seed(SEED)

KAGGLE_INPUT = Path(os.environ.get("KAGGLE_INPUT_DIR", "/kaggle/input"))
WORK_DIR = Path(os.environ.get("KAGGLE_WORKING_DIR", "/kaggle/working"))
if not WORK_DIR.exists():
    WORK_DIR = Path.cwd() / "kaggle_working_fallback"
WORK_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = WORK_DIR / "data"
SKILL_DIR = DATA_DIR / "skills"
REPLAY_DIR = DATA_DIR / "replay"
REPORT_DIR = DATA_DIR / "reports"
AGENT_DIR = WORK_DIR / "agent"
for p in [DATA_DIR, SKILL_DIR, REPLAY_DIR, REPORT_DIR, AGENT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "seed": SEED,
    "input_root": str(KAGGLE_INPUT),
    "work_dir": str(WORK_DIR),
    "max_steps_per_env": int(os.environ.get("NINE_ARC_MAX_STEPS", "240")),
    "epsilon": float(os.environ.get("NINE_ARC_EPSILON", "0.055")),
    "games": os.environ.get("NINE_ARC_GAMES", "all"),
    "render_mode": os.environ.get("NINE_ARC_RENDER_MODE", "terminal-fast"),
    "source_url": os.environ.get("NINE_ARC_SOURCE_URL", "https://github.com/engine/arc-agi3-component-skill-prefill"),
    "run_scorecard": os.environ.get("NINE_ARC_RUN_SCORECARD", "1") == "1",
    "scan_max_mb_per_file": float(os.environ.get("NINE_ARC_SCAN_MAX_MB", "16")),
    "import_utility_paths": os.environ.get("NINE_ARC_IMPORT_UTIL_PATHS", "1") == "1",
}
print(json.dumps(CONFIG, indent=2))
print("KAGGLE_INPUT exists:", KAGGLE_INPUT.exists())


## 1. Official package import check

The notebook can still build the skill files and export `agent/my_agent.py` without the ARC package. The scorecard cell requires the official competition runtime.


In [ ]:
def try_import_arc_packages():
    try:
        import arc_agi
        from arc_agi import Arcade
        try:
            from arc_agi import Agent
        except Exception:
            Agent = object
        from arcengine import GameAction, GameState
        return True, arc_agi, Arcade, Agent, GameAction, GameState, None
    except Exception as e:
        return False, None, None, object, None, None, e

ARC_OK, arc_agi, Arcade, Agent, GameAction, GameState, ARC_IMPORT_ERROR = try_import_arc_packages()
print("ARC packages available:", ARC_OK)
if ARC_IMPORT_ERROR:
    print("ARC import error:", repr(ARC_IMPORT_ERROR))


## 2. Component and dataset discovery engine

This is the required pre-run search pass. It detects notebooks, datasets, utility scripts, archives, prior replay files, and existing skill libraries. Public code is parsed statically; it is not executed.


In [ ]:
@dataclass
class Skill:
    skill_id: str
    name: str
    description: str
    trigger_conditions: List[str]
    action_template: str
    confidence: float
    success_rate: float
    examples_count: int
    source_files: List[str] = field(default_factory=list)
    tags: List[str] = field(default_factory=list)
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class DetectedComponent:
    component_id: str
    kind: str
    name: str
    source_path: str
    score: float
    tags: List[str]
    evidence: Dict[str, Any]

class SkillLibrary:
    def __init__(self, save_path: Path):
        self.save_path = Path(save_path)
        self.skills: Dict[str, Skill] = {}
        self.load()

    def load(self):
        if not self.save_path.exists():
            return
        try:
            raw = json.loads(self.save_path.read_text())
            for sid, row in raw.items():
                row.setdefault("source_files", [])
                row.setdefault("tags", [])
                row.setdefault("metadata", {})
                self.skills[sid] = Skill(**row)
        except Exception as e:
            print("skill load failed", repr(e))
            self.skills = {}

    def save(self):
        self.save_path.parent.mkdir(parents=True, exist_ok=True)
        payload = {sid: asdict(s) for sid, s in sorted(self.skills.items())}
        self.save_path.write_text(json.dumps(payload, indent=2, sort_keys=True))

    def add_or_update(self, skill: Skill):
        old = self.skills.get(skill.skill_id)
        if old is None:
            self.skills[skill.skill_id] = skill
        else:
            total = max(1, old.examples_count + skill.examples_count)
            old.confidence = ((old.confidence * old.examples_count) + (skill.confidence * skill.examples_count)) / total
            old.success_rate = ((old.success_rate * old.examples_count) + (skill.success_rate * skill.examples_count)) / total
            old.examples_count = total
            old.source_files = sorted(set(old.source_files + skill.source_files))[:60]
            old.tags = sorted(set(old.tags + skill.tags))[:80]
            old.trigger_conditions = sorted(set(old.trigger_conditions + skill.trigger_conditions))[:120]
            merged = dict(old.metadata)
            merged.update(skill.metadata)
            old.metadata = merged
        self.save()

class InputComponentMiner:
    SOURCE_EXTS = {".py", ".ipynb", ".md", ".txt", ".json", ".jsonl", ".yaml", ".yml"}
    DATA_EXTS = {".csv", ".tsv", ".parquet", ".feather", ".pkl", ".pickle", ".npy", ".npz"}
    ARCHIVE_EXTS = {".zip", ".whl"}
    SKILL_KEYS = {"skill_id", "name", "description", "trigger_conditions", "action_template", "confidence", "success_rate", "examples_count"}
    ACTION_RE = re.compile(r"GameAction\.(ACTION\d+|RESET)|\bACTION\d+\b|\bRESET\b")
    GAME_RE = re.compile(r"\b([a-z]{2}\d{2}|[a-z]\d{2}[a-z]?)\b", re.I)
    SIGNALS = {
        "agent_contract": ["choose_action", "is_done", "Agent", "FrameData", "GameState", "available_actions", "action_space"],
        "action_api": ["GameAction", "ACTION1", "ACTION2", "ACTION3", "ACTION4", "ACTION5", "ACTION6", "set_data", "is_complex", "env.step"],
        "click_targeting": ["ACTION6", "click", "coordinate", "centroid", "bbox", "x", "y", "target"],
        "object_extraction": ["connected", "component", "object", "bbox", "centroid", "color", "nonzero", "visited", "queue"],
        "state_graph": ["state_hash", "visited", "frontier", "graph", "transition", "edge", "BFS", "DFS", "untried"],
        "replay_memory": ["replay", "experience", "buffer", "transitions", "jsonl", "sqlite", "changed", "progress"],
        "world_model": ["world model", "predict", "simulate", "verifier", "hypothesis", "delta", "effect", "forward"],
        "stagnation_recovery": ["stagnation", "same_hash", "no_change", "stuck", "reset", "GAME_OVER", "timeout"],
        "skill_library": ["SkillLibrary", "Skill", "skill_id", "trigger_conditions", "action_template", "success_rate"],
        "training_loop": ["scorecard", "create_scorecard", "close_scorecard", "get_environments", "arc.make", "episodes", "train"],
    }
    TEMPLATE_BY_KIND = {
        "agent_contract": "official_agent_hook_contract_choose_action_is_done",
        "action_api": "valid_action_space_and_env_step_api",
        "click_targeting": "action6_coordinate_targeting_from_salient_features",
        "object_extraction": "connected_component_object_salience_extraction",
        "state_graph": "state_hash_graph_explore_untried_then_best_delta",
        "replay_memory": "transition_replay_prefer_changed_progress_edges",
        "world_model": "verify_action_delta_before_committing_plan",
        "stagnation_recovery": "rotate_strategy_and_reset_on_repeated_noop",
        "skill_library": "retrieve_skill_template_by_runtime_context",
        "training_loop": "scorecard_aware_training_and_eval_loop",
        "dataset_action_prior": "prefer_actions_with_positive_transition_evidence",
        "existing_skill_library": "reuse_attached_skill_library_priors",
    }

    def __init__(self, root: Path, max_mb_per_file: float = 16.0):
        self.root = Path(root)
        self.max_bytes = int(max_mb_per_file * 1024 * 1024)
        self.inventory: List[Dict[str, Any]] = []
        self.components: List[DetectedComponent] = []
        self.failures: List[Dict[str, str]] = []
        self.dataset_summaries: List[Dict[str, Any]] = []
        self.existing_skills: List[Skill] = []
        self.transition_priors: Dict[str, Any] = {}
        self.utility_paths: List[str] = []

    def file_kind(self, path: Path) -> str:
        s = path.suffix.lower()
        if s == ".ipynb": return "notebook"
        if s == ".py": return "python_utility"
        if s in {".md", ".txt", ".yaml", ".yml"}: return "text_or_config"
        if s in {".json", ".jsonl"}: return "json_dataset_or_source"
        if s in self.DATA_EXTS: return "tabular_or_binary_dataset"
        if s in self.ARCHIVE_EXTS: return "archive_or_wheel_utility"
        return "other"

    def sha12(self, b: bytes) -> str:
        return hashlib.sha1(b).hexdigest()[:12]

    def inventory_inputs(self) -> List[Dict[str, Any]]:
        rows = []
        if not self.root.exists():
            self.inventory = []
            return []
        for path in sorted(self.root.rglob("*")):
            if path.is_dir():
                # potential utility package root
                if any((path / marker).exists() for marker in ["__init__.py", "setup.py", "pyproject.toml"]):
                    self.utility_paths.append(str(path.parent))
                continue
            try:
                st = path.stat()
                rows.append({
                    "path": str(path),
                    "relpath": str(path.relative_to(self.root)),
                    "suffix": path.suffix.lower(),
                    "kind": self.file_kind(path),
                    "size_bytes": st.st_size,
                    "size_mb": round(st.st_size / 1024 / 1024, 4),
                })
                if path.suffix.lower() == ".py":
                    self.utility_paths.append(str(path.parent))
                if path.suffix.lower() in {".zip", ".whl"}:
                    self.utility_paths.append(str(path))
            except Exception as e:
                self.failures.append({"path": str(path), "error": repr(e)})
        self.inventory = rows
        return rows

    def source_units_from_path(self, path: Path) -> List[Tuple[str, str]]:
        suffix = path.suffix.lower()
        try:
            if path.stat().st_size > self.max_bytes:
                return []
            if suffix == ".ipynb":
                nb = json.loads(path.read_text(errors="ignore"))
                out = []
                for i, cell in enumerate(nb.get("cells", [])):
                    src = cell.get("source", "")
                    if isinstance(src, list):
                        src = "".join(src)
                    if src.strip():
                        out.append((f"{path}::cell{i}:{cell.get('cell_type')}", src))
                return out
            if suffix in self.SOURCE_EXTS:
                return [(str(path), path.read_text(errors="ignore"))]
            if suffix in self.ARCHIVE_EXTS:
                out = []
                with zipfile.ZipFile(path) as zf:
                    for info in zf.infolist():
                        name = info.filename
                        ext = Path(name).suffix.lower()
                        if ext in {".py", ".md", ".txt", ".json", ".yaml", ".yml"} and info.file_size <= min(self.max_bytes, 2_000_000):
                            try:
                                data = zf.read(info).decode("utf-8", errors="ignore")
                                out.append((f"{path}::{name}", data))
                            except Exception:
                                pass
                return out
        except Exception as e:
            self.failures.append({"path": str(path), "error": repr(e)})
        return []

    def preview_dataset(self, path: Path):
        suffix = path.suffix.lower()
        summary = {"path": str(path), "suffix": suffix, "kind": self.file_kind(path)}
        try:
            if path.stat().st_size > self.max_bytes:
                summary["skipped"] = "too_large"
                self.dataset_summaries.append(summary)
                return
            if suffix in {".csv", ".tsv"}:
                delim = "\t" if suffix == ".tsv" else ","
                with path.open("r", errors="ignore", newline="") as f:
                    reader = csv.reader(f, delimiter=delim)
                    rows = []
                    for i, row in enumerate(reader):
                        rows.append(row)
                        if i >= 5: break
                header = rows[0] if rows else []
                summary.update({"columns": header, "preview_rows": rows[:3]})
                self._detect_transition_table(path, header, rows[1:])
            elif suffix == ".json":
                obj = json.loads(path.read_text(errors="ignore"))
                summary.update({"json_type": type(obj).__name__})
                if isinstance(obj, dict):
                    summary["top_keys"] = list(obj.keys())[:40]
                    self._ingest_skill_json(path, obj)
                elif isinstance(obj, list):
                    summary["list_len_preview"] = min(len(obj), 20)
                    self._detect_transition_rows(path, obj[:200])
            elif suffix == ".jsonl":
                rows = []
                with path.open("r", errors="ignore") as f:
                    for i, line in enumerate(f):
                        if i >= 500: break
                        line = line.strip()
                        if not line: continue
                        try: rows.append(json.loads(line))
                        except Exception: pass
                summary["jsonl_rows_previewed"] = len(rows)
                self._detect_transition_rows(path, rows)
            elif suffix == ".parquet":
                try:
                    import pandas as pd
                    df = pd.read_parquet(path)
                    summary.update({"columns": list(df.columns), "rows_previewed": int(min(len(df), 1000))})
                    self._detect_transition_rows(path, df.head(500).to_dict("records"))
                except Exception as e:
                    summary["parquet_preview_error"] = repr(e)
        except Exception as e:
            summary["error"] = repr(e)
            self.failures.append({"path": str(path), "error": repr(e)})
        self.dataset_summaries.append(summary)

    def _ingest_skill_json(self, path: Path, obj: Any):
        if not isinstance(obj, dict):
            return
        # Either {skill_id: skill_dict} or {"skills": [.../dict]}
        candidates = []
        if "skills" in obj:
            val = obj["skills"]
            if isinstance(val, dict): candidates.extend(val.values())
            if isinstance(val, list): candidates.extend(val)
        else:
            candidates.extend(obj.values())
        for row in candidates:
            if not isinstance(row, dict):
                continue
            if len(self.SKILL_KEYS.intersection(row.keys())) >= 6:
                try:
                    row = dict(row)
                    row.setdefault("source_files", [])
                    row.setdefault("tags", [])
                    row.setdefault("metadata", {})
                    row["source_files"] = sorted(set(row.get("source_files", []) + [str(path)]))
                    row["metadata"] = dict(row.get("metadata", {}), ingested_from=str(path))
                    self.existing_skills.append(Skill(**{k: row[k] for k in Skill.__dataclass_fields__.keys() if k in row}))
                except Exception as e:
                    self.failures.append({"path": str(path), "error": "skill_ingest " + repr(e)})

    def _detect_transition_table(self, path: Path, header: List[str], rows: List[List[str]]):
        normalized = [h.strip().lower() for h in header]
        if any(c in normalized for c in ["action", "action_name", "gameaction"]) and any(c in normalized for c in ["changed", "win", "progress", "level_progress", "is_win"]):
            idx = {name: i for i, name in enumerate(normalized)}
            dict_rows = []
            for r in rows[:100]:
                dict_rows.append({h: (r[i] if i < len(r) else None) for h, i in idx.items()})
            self._detect_transition_rows(path, dict_rows)

    def _detect_transition_rows(self, path: Path, rows: List[Any]):
        action_counter = Counter()
        good_counter = Counter()
        bad_counter = Counter()
        game_counter = Counter()
        for row in rows:
            if not isinstance(row, dict):
                continue
            action = row.get("action") or row.get("action_name") or row.get("gameaction") or row.get("chosen_action")
            if not action:
                continue
            action = str(action).split(".")[-1]
            action_counter[action] += 1
            changed = str(row.get("changed", row.get("grid_changed", ""))).lower() in {"true", "1", "yes"}
            progress = bool(row.get("progress") or row.get("level_progress") or row.get("is_win") or row.get("win"))
            game = row.get("game_id") or row.get("game") or "unknown"
            game_counter[str(game)] += 1
            if changed or progress:
                good_counter[action] += 1
            if str(row.get("game_over", row.get("bad", ""))).lower() in {"true", "1", "yes"}:
                bad_counter[action] += 1
        if action_counter:
            pri = self.transition_priors.setdefault("sources", [])
            pri.append({
                "path": str(path),
                "actions": dict(action_counter.most_common()),
                "positive_actions": dict(good_counter.most_common()),
                "bad_actions": dict(bad_counter.most_common()),
                "games": dict(game_counter.most_common(25)),
            })

    def score_source_component(self, source_name: str, src: str) -> List[DetectedComponent]:
        comps = []
        low = src.lower()
        actions = [m.group(0).replace("GameAction.", "") for m in self.ACTION_RE.finditer(src)]
        games = [g.lower() for g in self.GAME_RE.findall(src)]
        funcs, classes = [], []
        try:
            tree = ast.parse(src)
            for node in ast.walk(tree):
                if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                    funcs.append(node.name)
                elif isinstance(node, ast.ClassDef):
                    classes.append(node.name)
        except Exception:
            pass
        for kind, terms in self.SIGNALS.items():
            term_hits = [t for t in terms if t.lower() in low]
            if not term_hits:
                continue
            # Solver value: hooks/API > graph/replay/world model > object/click > docs-only terms.
            base_weight = {
                "agent_contract": 4.0, "action_api": 3.5, "state_graph": 4.5, "replay_memory": 4.0,
                "world_model": 4.2, "click_targeting": 3.4, "object_extraction": 3.2,
                "stagnation_recovery": 3.0, "skill_library": 2.8, "training_loop": 2.6,
            }.get(kind, 2.0)
            score = base_weight * len(term_hits)
            score += 0.35 * len(actions)
            score += 0.6 * sum(1 for f in funcs if f in {"choose_action", "is_done", "extract_objects", "get_grid_from_obs", "normalize_available_actions"})
            score += 0.5 * sum(1 for c in classes if c.lower().endswith(("agent", "policy", "memory", "solver")))
            cid = hashlib.sha1(f"{kind}|{source_name}|{term_hits}|{actions[:8]}".encode()).hexdigest()[:12]
            comps.append(DetectedComponent(
                component_id=f"component_{kind}_{cid}",
                kind=kind,
                name=f"{kind} from {Path(source_name.split('::')[0]).name}",
                source_path=source_name,
                score=round(score, 4),
                tags=sorted(set([kind] + term_hits + actions[:12] + funcs[:10] + classes[:10] + games[:10]))[:80],
                evidence={"term_hits": term_hits, "actions": actions[:40], "functions": funcs[:40], "classes": classes[:40], "games": games[:25]},
            ))
        return comps

    def scan(self):
        rows = self.inventory_inputs()
        for row in rows:
            path = Path(row["path"])
            suffix = path.suffix.lower()
            if row["kind"] in {"tabular_or_binary_dataset", "json_dataset_or_source"}:
                self.preview_dataset(path)
            if suffix in self.SOURCE_EXTS or suffix in self.ARCHIVE_EXTS:
                for source_name, src in self.source_units_from_path(path):
                    self.components.extend(self.score_source_component(source_name, src))
        self.components.sort(key=lambda c: c.score, reverse=True)
        return self

    def build_skills(self) -> List[Skill]:
        skills: Dict[str, Skill] = {}
        # 1. Pass through existing skill libraries.
        for sk in self.existing_skills:
            sk.skill_id = "ingested_" + sk.skill_id if not sk.skill_id.startswith("ingested_") else sk.skill_id
            sk.confidence = min(0.95, max(sk.confidence, 0.68))
            sk.tags = sorted(set(sk.tags + ["ingested_skill_library"]))
            skills[sk.skill_id] = sk
        # 2. Aggregate static components by kind.
        by_kind: Dict[str, List[DetectedComponent]] = defaultdict(list)
        for c in self.components:
            by_kind[c.kind].append(c)
        for kind, comps in by_kind.items():
            top = comps[:12]
            score_sum = sum(c.score for c in top)
            tags = sorted(set(t for c in top for t in c.tags))[:100]
            srcs = sorted(set(c.source_path.split("::")[0] for c in top))[:40]
            examples = max(1, int(sum(max(1, len(c.evidence.get("term_hits", []))) for c in top)))
            conf = min(0.92, 0.42 + 0.018 * min(score_sum, 22) + 0.025 * min(len(srcs), 8))
            sid = "skill_component_" + kind + "_" + hashlib.sha1(json.dumps([c.component_id for c in top], sort_keys=True).encode()).hexdigest()[:10]
            skills[sid] = Skill(
                skill_id=sid,
                name=f"component_{kind}_skill",
                description=f"Detected and ranked {kind} solver component from attached Kaggle inputs.",
                trigger_conditions=sorted(set([kind] + tags + self.TEMPLATE_BY_KIND.get(kind, kind).split("_")))[:140],
                action_template=self.TEMPLATE_BY_KIND.get(kind, f"component_{kind}_template"),
                confidence=conf,
                success_rate=0.50,
                examples_count=examples,
                source_files=srcs,
                tags=tags,
                metadata={"component_kind": kind, "component_ids": [c.component_id for c in top], "score_sum": score_sum},
            )
        # 3. Transition/action priors from datasets.
        action_counts = Counter()
        positive_counts = Counter()
        bad_counts = Counter()
        source_files = []
        for src in self.transition_priors.get("sources", []):
            source_files.append(src["path"])
            action_counts.update(src.get("actions", {}))
            positive_counts.update(src.get("positive_actions", {}))
            bad_counts.update(src.get("bad_actions", {}))
        if action_counts:
            common = [a for a, _ in action_counts.most_common(10)]
            pos = [a for a, _ in positive_counts.most_common(10)]
            sid = "skill_dataset_action_prior_" + hashlib.sha1(json.dumps(dict(action_counts), sort_keys=True).encode()).hexdigest()[:10]
            skills[sid] = Skill(
                skill_id=sid,
                name="dataset_transition_action_prior",
                description="Action prior derived from attached replay/transition datasets.",
                trigger_conditions=sorted(set(common + pos + ["transition", "action_space", "replay", "progress", "changed"])),
                action_template="prefer_dataset_positive_actions_then_untried_edges",
                confidence=0.74,
                success_rate=0.52,
                examples_count=sum(action_counts.values()),
                source_files=sorted(set(source_files))[:50],
                tags=sorted(set(["dataset", "transition_prior", "replay"] + common + pos)),
                metadata={"action_counts": dict(action_counts.most_common(20)), "positive_counts": dict(positive_counts.most_common(20)), "bad_counts": dict(bad_counts.most_common(20))},
            )
        return sorted(skills.values(), key=lambda s: (s.confidence, s.examples_count), reverse=True)

    def write_reports(self, report_dir: Path):
        report_dir.mkdir(parents=True, exist_ok=True)
        (report_dir / "input_inventory.json").write_text(json.dumps(self.inventory, indent=2))
        (report_dir / "component_rankings.json").write_text(json.dumps([asdict(c) for c in self.components], indent=2))
        (report_dir / "dataset_summaries.json").write_text(json.dumps(self.dataset_summaries, indent=2, default=str))
        (report_dir / "scan_failures.json").write_text(json.dumps(self.failures, indent=2))
        (report_dir / "transition_priors.json").write_text(json.dumps(self.transition_priors, indent=2))


def seed_builtin_skills() -> List[Skill]:
    return [
        Skill("skill_builtin_graph_memory", "graph_memory_policy", "Explore untried state/action edges, then exploit changed/progress edges.", ["graph", "state_hash", "untried", "transition", "progress", "edge"], "state_hash_graph_explore_untried_then_best_delta", 0.72, 0.50, 1, tags=["graph", "replay", "policy"]),
        Skill("skill_builtin_action6_salience", "action6_salient_clicks", "Use ACTION6 only when valid and click object centroids, corners, samples, and center.", ["ACTION6", "click", "coordinate", "object", "centroid", "bbox", "color"], "rank_action6_targets_by_salient_object_centroid_corner_center", 0.68, 0.50, 1, tags=["ACTION6", "object", "click"]),
        Skill("skill_builtin_valid_action_space", "valid_action_space_only", "Never invent actions; normalize env.action_space / available_actions and reset only as recovery.", ["available_actions", "action_space", "RESET", "valid", "complex"], "valid_actions_only_never_fake_coordinates", 0.64, 0.50, 1, tags=["safety", "contract"]),
        Skill("skill_builtin_stagnation_recovery", "stagnation_recovery", "Rotate click target strategy after repeated no-op states; reset if terminal/recovery action is exposed.", ["stagnation", "same_hash", "no_change", "GAME_OVER", "RESET"], "rotate_strategy_after_no_change_reset_if_available", 0.62, 0.50, 1, tags=["reset", "stagnation"]),
    ]


## 3. Run discovery, rank components, and prebuild skills

This cell is the core requirement: it searches all Kaggle inputs and uses the best detected components to seed the runtime skill library.


In [ ]:
miner = InputComponentMiner(KAGGLE_INPUT, max_mb_per_file=CONFIG["scan_max_mb_per_file"]).scan()
miner.write_reports(REPORT_DIR)

# Optional import path discovery for attached utility packages/wheels. This does not execute their code.
if CONFIG["import_utility_paths"]:
    for p in sorted(set(miner.utility_paths)):
        if p and p not in sys.path:
            sys.path.insert(0, p)

skill_path = SKILL_DIR / "skill_library.json"
library = SkillLibrary(skill_path)
for sk in seed_builtin_skills():
    library.add_or_update(sk)
for sk in miner.build_skills():
    library.add_or_update(sk)

manifest = {
    "created_at": time.time(),
    "input_files": len(miner.inventory),
    "detected_components": len(miner.components),
    "dataset_summaries": len(miner.dataset_summaries),
    "existing_skills_ingested": len(miner.existing_skills),
    "skills_total": len(library.skills),
    "utility_paths_added": sorted(set(miner.utility_paths))[:100],
    "top_components": [asdict(c) for c in miner.components[:25]],
    "top_skills": [asdict(s) for s in sorted(library.skills.values(), key=lambda s: (s.confidence, s.examples_count), reverse=True)[:25]],
}
(REPORT_DIR / "component_skill_manifest.json").write_text(json.dumps(manifest, indent=2, default=str))

print("input files:", len(miner.inventory))
print("components detected:", len(miner.components))
print("existing skills ingested:", len(miner.existing_skills))
print("skills saved:", skill_path)
print("skill count:", len(library.skills))
print("utility paths added:", len(set(miner.utility_paths)))
print("\nTop components:")
for c in miner.components[:10]:
    print(f"- {c.kind:22s} score={c.score:6.2f} source={c.source_path[:120]}")
print("\nTop skills:")
for s in sorted(library.skills.values(), key=lambda s: (s.confidence, s.examples_count), reverse=True)[:10]:
    print(f"- {s.skill_id} conf={s.confidence:.3f} examples={s.examples_count} template={s.action_template}")


## 4. Export `agent/my_agent.py`

This is the official-starter-compatible agent file. It reads the prebuilt `data/skills/skill_library.json` generated above and uses graph memory + object/click candidate generation during play.


In [ ]:
AGENT_SOURCE = '\nimport os, json, time, random, hashlib, math, re\nfrom pathlib import Path\nfrom dataclasses import dataclass, asdict, field\nfrom collections import defaultdict, deque, Counter\nfrom typing import Any, Dict, List, Optional, Tuple, Set, Iterable\n\ntry:\n    from arc_agi import Agent\nexcept Exception:\n    class Agent:  # offline dry-run placeholder\n        pass\ntry:\n    from arcengine import GameAction, GameState\nexcept Exception:\n    class _FakeAction:\n        def __init__(self, name, value):\n            self.name = name\n            self.value = value\n            self.reasoning = None\n            self._data = {}\n        def is_complex(self): return self.name == "ACTION6"\n        def is_simple(self): return not self.is_complex()\n        def set_data(self, data): self._data = data\n        def __str__(self): return f"GameAction.{self.name}"\n        def __repr__(self): return self.__str__()\n    class GameAction:\n        RESET = _FakeAction("RESET", 0)\n        ACTION1 = _FakeAction("ACTION1", 1)\n        ACTION2 = _FakeAction("ACTION2", 2)\n        ACTION3 = _FakeAction("ACTION3", 3)\n        ACTION4 = _FakeAction("ACTION4", 4)\n        ACTION5 = _FakeAction("ACTION5", 5)\n        ACTION6 = _FakeAction("ACTION6", 6)\n        ACTION7 = _FakeAction("ACTION7", 7)\n        @classmethod\n        def __iter__(cls):\n            return iter([cls.RESET, cls.ACTION1, cls.ACTION2, cls.ACTION3, cls.ACTION4, cls.ACTION5, cls.ACTION6, cls.ACTION7])\n    class GameState:\n        WIN = "WIN"\n        GAME_OVER = "GAME_OVER"\n        NOT_FINISHED = "NOT_FINISHED"\n        NOT_PLAYED = "NOT_PLAYED"\n        NOT_STARTED = "NOT_STARTED"\n\n@dataclass\nclass Skill:\n    skill_id: str\n    name: str\n    description: str\n    trigger_conditions: List[str]\n    action_template: str\n    confidence: float\n    success_rate: float\n    examples_count: int\n    source_files: List[str] = field(default_factory=list)\n    tags: List[str] = field(default_factory=list)\n    metadata: Dict[str, Any] = field(default_factory=dict)\n\nclass SkillLibrary:\n    def __init__(self, save_path: str = None):\n        default_path = Path(os.environ.get("NINE_ARC_SKILL_PATH", "/kaggle/working/data/skills/skill_library.json"))\n        self.save_path = Path(save_path) if save_path else default_path\n        self.skills: Dict[str, Skill] = {}\n        self.load()\n\n    def load(self):\n        if not self.save_path.exists():\n            return\n        try:\n            data = json.loads(self.save_path.read_text())\n            for sid, row in data.items():\n                self.skills[sid] = Skill(**row)\n        except Exception:\n            self.skills = {}\n\n    def save(self):\n        try:\n            self.save_path.parent.mkdir(parents=True, exist_ok=True)\n            self.save_path.write_text(json.dumps({sid: asdict(s) for sid, s in self.skills.items()}, indent=2, sort_keys=True))\n        except Exception:\n            pass\n\n    def retrieve(self, query: str, top_k: int = 5) -> List[Skill]:\n        if not self.skills:\n            return []\n        q = query.lower()\n        scored = []\n        for s in self.skills.values():\n            score = s.confidence * 0.6 + s.success_rate * 0.4\n            for term in s.trigger_conditions + s.tags + [s.name, s.action_template]:\n                t = str(term).lower()\n                if t and t in q:\n                    score += 1.0\n            scored.append((score, s))\n        scored.sort(key=lambda x: x[0], reverse=True)\n        return [s for _, s in scored[:top_k]]\n\n    def record_outcome(self, skill_id: str, success: bool):\n        s = self.skills.get(skill_id)\n        if not s:\n            return\n        n = max(0, s.examples_count)\n        reward = 1.0 if success else 0.0\n        s.success_rate = ((s.success_rate * n) + reward) / (n + 1)\n        s.confidence = min(1.0, max(0.05, s.confidence + (0.04 if success else -0.055)))\n        s.examples_count = n + 1\n        self.save()\n\ndef action_name(action: Any) -> str:\n    return getattr(action, "name", str(action).split(".")[-1])\n\ndef action_value(action: Any) -> int:\n    v = getattr(action, "value", None)\n    if isinstance(v, int): return v\n    m = re.search(r"ACTION(\\d+)", action_name(action))\n    if m: return int(m.group(1))\n    if action_name(action) == "RESET": return 0\n    return 999\n\ndef is_complex_action(action: Any) -> bool:\n    try:\n        return bool(action.is_complex())\n    except Exception:\n        return action_name(action) == "ACTION6"\n\ndef pack_action(action: Any, data: Optional[Dict[str, Any]], reasoning: Optional[Dict[str, Any]] = None):\n    data = data or {}\n    reasoning = reasoning or {}\n    if data:\n        try:\n            action.set_data(data)\n        except Exception:\n            try:\n                action.data = data\n            except Exception:\n                pass\n    try:\n        action.reasoning = reasoning\n    except Exception:\n        pass\n    return action\n\ndef get_grid_from_obs(obs: Any) -> List[List[int]]:\n    frame = getattr(obs, "frame", None)\n    if frame is None:\n        frame = getattr(obs, "grid", None)\n    if frame is None:\n        return []\n    # FrameDataRaw.frame can be [frame_count][64][64]. Use latest.\n    try:\n        if frame and isinstance(frame, list) and isinstance(frame[0], list) and frame[0] and isinstance(frame[0][0], list):\n            return frame[-1]\n    except Exception:\n        pass\n    return frame\n\ndef normalize_available_actions(obs: Any = None, env: Any = None) -> List[Any]:\n    raw = []\n    if env is not None:\n        try:\n            raw = list(env.action_space or [])\n        except Exception:\n            raw = []\n    if not raw and obs is not None:\n        raw = getattr(obs, "available_actions", []) or []\n    converted = []\n    for a in raw:\n        if hasattr(a, "name"):\n            converted.append(a)\n        elif isinstance(a, int):\n            name = "RESET" if a == 0 else f"ACTION{a}"\n            if hasattr(GameAction, name):\n                converted.append(getattr(GameAction, name))\n        elif isinstance(a, str):\n            name = a.split(".")[-1]\n            if hasattr(GameAction, name):\n                converted.append(getattr(GameAction, name))\n    if converted:\n        return sorted(converted, key=action_value)\n    # Conservative fallback: RESET only. Standalone runner should pass env.action_space.\n    return [GameAction.RESET] if hasattr(GameAction, "RESET") else []\n\ndef grid_hash(grid: List[List[int]]) -> str:\n    if not grid:\n        return "empty"\n    h = hashlib.blake2b(digest_size=10)\n    for row in grid:\n        h.update(bytes([int(x) & 255 for x in row]))\n    return h.hexdigest()\n\ndef grid_delta_score(a: List[List[int]], b: List[List[int]]) -> int:\n    if not a or not b or len(a) != len(b):\n        return 0\n    score = 0\n    for r in range(min(len(a), len(b))):\n        ra, rb = a[r], b[r]\n        for c in range(min(len(ra), len(rb))):\n            if ra[c] != rb[c]:\n                score += 1\n    return score\n\ndef extract_objects(grid: List[List[int]]) -> List[Dict[str, Any]]:\n    if not grid or not grid[0]:\n        return []\n    H, W = len(grid), len(grid[0])\n    visited = [[False] * W for _ in range(H)]\n    objects = []\n    for r in range(H):\n        for c in range(W):\n            if visited[r][c] or grid[r][c] == 0:\n                continue\n            color = grid[r][c]\n            q = [(r, c)]\n            visited[r][c] = True\n            coords = []\n            while q:\n                rr, cc = q.pop()\n                coords.append((rr, cc))\n                for dr, dc in [(-1,0), (1,0), (0,-1), (0,1)]:\n                    nr, nc = rr + dr, cc + dc\n                    if 0 <= nr < H and 0 <= nc < W and not visited[nr][nc] and grid[nr][nc] == color:\n                        visited[nr][nc] = True\n                        q.append((nr, nc))\n            rows = [p[0] for p in coords]\n            cols = [p[1] for p in coords]\n            bbox = (min(rows), min(cols), max(rows), max(cols))\n            objects.append({\n                "color": int(color),\n                "pixels": coords,\n                "size": len(coords),\n                "bbox": bbox,\n                "centroid": (sum(cols) / len(cols), sum(rows) / len(rows)),\n            })\n    objects.sort(key=lambda o: (-o["size"], o["bbox"]))\n    return objects[:128]\n\ndef salient_click_targets(grid: List[List[int]]) -> List[Tuple[int, int, str]]:\n    if not grid or not grid[0]:\n        return [(32, 32, "empty_center")]\n    H, W = len(grid), len(grid[0])\n    targets: List[Tuple[int, int, str]] = []\n    objects = extract_objects(grid)\n    for i, obj in enumerate(objects[:10]):\n        min_r, min_c, max_r, max_c = obj["bbox"]\n        cx, cy = obj["centroid"]\n        candidates = [\n            (round(cx), round(cy), f"obj{i}_centroid"),\n            ((min_c + max_c) // 2, (min_r + max_r) // 2, f"obj{i}_bbox_center"),\n            (min_c, min_r, f"obj{i}_top_left"),\n            (max_c, max_r, f"obj{i}_bottom_right"),\n        ]\n        for x, y, label in candidates:\n            x = max(0, min(W - 1, int(x)))\n            y = max(0, min(H - 1, int(y)))\n            targets.append((x, y, label))\n    colored = [(c, r) for r in range(H) for c in range(W) if grid[r][c] != 0]\n    if colored:\n        for j in range(min(8, len(colored))):\n            x, y = colored[(j * max(1, len(colored)//8)) % len(colored)]\n            targets.append((x, y, f"colored_sample_{j}"))\n    targets.extend([(W//2, H//2, "center"), (0, 0, "origin"), (W-1, H-1, "far_corner")])\n    dedup = []\n    seen = set()\n    for x, y, label in targets:\n        key = (x, y)\n        if key not in seen:\n            seen.add(key)\n            dedup.append((x, y, label))\n    return dedup\n\nclass TransitionMemory:\n    def __init__(self, path: str = None):\n        default = Path(os.environ.get("NINE_ARC_REPLAY_PATH", "/kaggle/working/data/replay/transitions.jsonl"))\n        self.path = Path(path) if path else default\n        self.path.parent.mkdir(parents=True, exist_ok=True)\n        self.edges: Dict[str, Dict[str, Dict[str, Any]]] = defaultdict(dict)\n        self.action_stats: Dict[str, Dict[str, float]] = defaultdict(lambda: {"n": 0, "changed": 0, "progress": 0, "win": 0, "game_over": 0})\n        self.load()\n\n    def key_for(self, action: Any, data: Optional[Dict[str, Any]]) -> str:\n        return json.dumps({"a": action_name(action), "d": data or {}}, sort_keys=True)\n\n    def load(self):\n        if not self.path.exists():\n            return\n        try:\n            for line in self.path.read_text(errors="ignore").splitlines()[-20000:]:\n                if not line.strip():\n                    continue\n                row = json.loads(line)\n                sh = row.get("state_before")\n                ak = row.get("action_key")\n                if sh and ak:\n                    self.edges[sh][ak] = row\n                    st = self.action_stats[row.get("action", ak)]\n                    st["n"] += 1\n                    st["changed"] += int(bool(row.get("changed")))\n                    st["progress"] += int(bool(row.get("progress")))\n                    st["win"] += int(bool(row.get("win")))\n                    st["game_over"] += int(bool(row.get("game_over")))\n        except Exception:\n            pass\n\n    def add(self, game_id: str, before: List[List[int]], action: Any, data: Dict[str, Any], after: List[List[int]], before_level: int, after_level: int, state: Any):\n        action_key = self.key_for(action, data)\n        delta = grid_delta_score(before, after)\n        progress = int(after_level) > int(before_level)\n        state_name = getattr(state, "name", str(state))\n        row = {\n            "ts": time.time(),\n            "game_id": game_id,\n            "state_before": grid_hash(before),\n            "state_after": grid_hash(after),\n            "action": action_name(action),\n            "action_key": action_key,\n            "action_data": data or {},\n            "delta": delta,\n            "changed": delta > 0,\n            "level_before": int(before_level),\n            "level_after": int(after_level),\n            "progress": progress,\n            "win": state_name == "WIN",\n            "game_over": state_name == "GAME_OVER",\n        }\n        self.edges[row["state_before"]][action_key] = row\n        st = self.action_stats[row["action"]]\n        st["n"] += 1\n        st["changed"] += int(row["changed"])\n        st["progress"] += int(row["progress"])\n        st["win"] += int(row["win"])\n        st["game_over"] += int(row["game_over"])\n        try:\n            with self.path.open("a") as f:\n                f.write(json.dumps(row, sort_keys=True) + "\\n")\n        except Exception:\n            pass\n        return row\n\n    def score_candidate(self, state_hash: str, action: Any, data: Dict[str, Any]) -> float:\n        ak = self.key_for(action, data)\n        if ak not in self.edges.get(state_hash, {}):\n            return 100.0 + random.random()  # untried edge exploration\n        row = self.edges[state_hash][ak]\n        score = 0.0\n        score += 25.0 if row.get("progress") else 0.0\n        score += 12.0 if row.get("win") else 0.0\n        score += min(10.0, float(row.get("delta", 0)) / 20.0)\n        score -= 20.0 if row.get("game_over") else 0.0\n        score -= 2.0\n        return score\n\n    def action_prior(self, action: Any) -> float:\n        name = action_name(action)\n        st = self.action_stats.get(name)\n        if not st or not st["n"]:\n            return 0.0\n        n = st["n"]\n        return (st["changed"] / n) * 2.0 + (st["progress"] / n) * 8.0 + (st["win"] / n) * 15.0 - (st["game_over"] / n) * 10.0\n\nclass SkillGraphPolicy:\n    def __init__(self, skill_library: SkillLibrary, memory: TransitionMemory, epsilon: float = None):\n        self.library = skill_library\n        self.memory = memory\n        self.epsilon = float(os.environ.get("NINE_ARC_EPSILON", "0.055")) if epsilon is None else float(epsilon)\n        self.strategy_rotation = 0\n        self.last_hash = None\n        self.no_change_steps = 0\n\n    def build_candidates(self, grid: List[List[int]], actions: List[Any]) -> List[Tuple[Any, Dict[str, Any], str]]:\n        candidates: List[Tuple[Any, Dict[str, Any], str]] = []\n        action6s = [a for a in actions if action_name(a) == "ACTION6" or is_complex_action(a)]\n        simple = [a for a in actions if not is_complex_action(a) and action_name(a) != "RESET"]\n        # Coordinate actions first when available, but cap candidate count.\n        if action6s:\n            targets = salient_click_targets(grid)\n            offset = self.strategy_rotation % max(1, len(targets))\n            targets = targets[offset:] + targets[:offset]\n            for x, y, label in targets[:24]:\n                for a in action6s:\n                    candidates.append((a, {"x": int(x), "y": int(y)}, label))\n        # Simple actions in stable numeric order.\n        for a in sorted(simple, key=action_value):\n            candidates.append((a, {}, f"simple_{action_name(a)}"))\n        # RESET last, only as recovery.\n        for a in actions:\n            if action_name(a) == "RESET":\n                candidates.append((a, {}, "reset_recovery"))\n        return candidates\n\n    def choose(self, game_id: str, latest_frame: Any, env: Any = None, step_index: int = 0) -> Tuple[Any, Dict[str, Any], Dict[str, Any], Optional[Skill]]:\n        grid = get_grid_from_obs(latest_frame)\n        actions = normalize_available_actions(latest_frame, env)\n        if not actions:\n            return GameAction.RESET, {}, {"thought": "No actions exposed; reset fallback."}, None\n        state_h = grid_hash(grid)\n        current_level = int(getattr(latest_frame, "levels_completed", getattr(latest_frame, "level_index", 0)) or 0)\n        objects = extract_objects(grid)\n        q = f"game={game_id} level={current_level} objects={len(objects)} colors={len(set(v for row in grid for v in row)) if grid else 0} action_space={\' \'.join(action_name(a) for a in actions)} graph object click replay reset policy"\n        skills = self.library.retrieve(q, top_k=3)\n        active_skill = skills[0] if skills else None\n        candidates = self.build_candidates(grid, actions)\n        if not candidates:\n            a = actions[0]\n            d = {"x": 32, "y": 32} if is_complex_action(a) else {}\n            return a, d, {"thought": "Fallback first valid action."}, active_skill\n        if random.random() < self.epsilon:\n            a, d, why = random.choice(candidates[:max(1, min(len(candidates), 12))])\n            return a, d, {"thought": f"epsilon exploration: {why}", "skill": active_skill.skill_id if active_skill else None}, active_skill\n        scored = []\n        for i, (a, d, why) in enumerate(candidates):\n            score = self.memory.score_candidate(state_h, a, d)\n            score += self.memory.action_prior(a)\n            # Mild public-notebook skill influence, not hard-coded game logic.\n            if active_skill:\n                tmpl = active_skill.action_template.lower()\n                if "action6" in tmpl and (action_name(a) == "ACTION6" or is_complex_action(a)):\n                    score += active_skill.confidence * 3\n                if "reset" in tmpl and why.startswith("reset"):\n                    score += active_skill.confidence * 1.5\n                if "untried" in tmpl:\n                    score += active_skill.confidence * 1.0\n            # Prefer non-reset early.\n            if action_name(a) == "RESET" and step_index < 12:\n                score -= 50\n            # Deterministic tie break: earlier candidates are more salient.\n            score -= i * 0.001\n            scored.append((score, a, d, why))\n        scored.sort(key=lambda x: x[0], reverse=True)\n        best_score, a, d, why = scored[0]\n        return a, d, {"thought": f"skill_graph_policy chose {action_name(a)} via {why}; score={best_score:.3f}", "skill": active_skill.skill_id if active_skill else None}, active_skill\n\nclass MyAgent(Agent):\n    """Official-starter-compatible ARC-AGI-3 agent using mined SkillLibrary + transition graph policy."""\n    def __init__(self):\n        super().__init__()\n        self.game_id = os.environ.get("NINE_ARC_CURRENT_GAME", "unknown")\n        self.library = SkillLibrary()\n        self.memory = TransitionMemory()\n        self.policy = SkillGraphPolicy(self.library, self.memory)\n        self.last_grid: List[List[int]] = []\n        self.last_action: Any = None\n        self.last_action_data: Dict[str, Any] = {}\n        self.last_level: int = 0\n        self.step_index = 0\n        self.active_skill: Optional[Skill] = None\n        self.max_steps = int(os.environ.get("NINE_ARC_MAX_STEPS", "240"))\n\n    def is_done(self, frames: List[Any], latest_frame: Any) -> bool:\n        state = getattr(latest_frame, "state", None)\n        state_name = getattr(state, "name", str(state))\n        if state_name in {"WIN", "GAME_OVER"}:\n            if self.active_skill:\n                self.library.record_outcome(self.active_skill.skill_id, state_name == "WIN")\n            return True\n        return self.step_index >= self.max_steps\n\n    def _commit_previous_transition(self, latest_frame: Any):\n        if not self.last_action or not self.last_grid:\n            return\n        grid = get_grid_from_obs(latest_frame)\n        level = int(getattr(latest_frame, "levels_completed", getattr(latest_frame, "level_index", 0)) or 0)\n        state = getattr(latest_frame, "state", None)\n        self.memory.add(self.game_id, self.last_grid, self.last_action, self.last_action_data, grid, self.last_level, level, state)\n        if grid_hash(grid) == grid_hash(self.last_grid):\n            self.policy.no_change_steps += 1\n            if self.policy.no_change_steps >= 3:\n                self.policy.strategy_rotation += 1\n                self.policy.no_change_steps = 0\n        else:\n            self.policy.no_change_steps = 0\n\n    def choose_action(self, frames: List[Any], latest_frame: Any):\n        self.step_index += 1\n        self._commit_previous_transition(latest_frame)\n        grid = get_grid_from_obs(latest_frame)\n        self.last_grid = grid\n        self.last_level = int(getattr(latest_frame, "levels_completed", getattr(latest_frame, "level_index", 0)) or 0)\n        # Official Agent hook cannot see env.action_space, so it uses latest_frame.available_actions.\n        action, data, reasoning, skill = self.policy.choose(self.game_id, latest_frame, env=None, step_index=self.step_index)\n        self.active_skill = skill\n        self.last_action = action\n        self.last_action_data = data or {}\n        return pack_action(action, data, reasoning)\n\n# Standalone runner helper uses env.action_space directly, which is stronger than the official hook.\ndef run_scorecard_with_agent(max_steps_per_env: int = 240, games: str = "all", render_mode: str = "terminal-fast", source_url: str = None):\n    from arc_agi import Arcade\n    arc = Arcade()\n    scorecard_id = arc.create_scorecard(\n        source_url=source_url or os.environ.get("NINE_ARC_SOURCE_URL", "https://github.com/engine/arc-agi3-skill-prefill"),\n        tags=["skill-prefill", "graph-memory", "public-notebook-mined"],\n    )\n    envs = arc.get_environments()\n    if games and games != "all":\n        wanted = {g.strip() for g in games.split(",") if g.strip()}\n        envs = [e for e in envs if getattr(e, "game_id", None) in wanted]\n    report = []\n    for meta in envs:\n        game_id = getattr(meta, "game_id", str(meta))\n        print("\\n=== GAME", game_id, "===")\n        os.environ["NINE_ARC_CURRENT_GAME"] = game_id\n        agent = MyAgent()\n        agent.game_id = game_id\n        env = arc.make(game_id, scorecard_id=scorecard_id, render_mode=render_mode)\n        obs = env.reset()\n        if obs is None:\n            report.append({"game_id": game_id, "status": "reset_failed", "steps": 0})\n            continue\n        frames = [obs]\n        latest = obs\n        steps = 0\n        while steps < max_steps_per_env and not agent.is_done(frames, latest):\n            # Standalone mode can use env.action_space every step.\n            agent.step_index += 1\n            agent._commit_previous_transition(latest)\n            grid = get_grid_from_obs(latest)\n            agent.last_grid = grid\n            agent.last_level = int(getattr(latest, "levels_completed", getattr(latest, "level_index", 0)) or 0)\n            action, data, reasoning, skill = agent.policy.choose(game_id, latest, env=env, step_index=agent.step_index)\n            agent.active_skill = skill\n            agent.last_action, agent.last_action_data = action, data or {}\n            obs = env.step(action, data=data or {}, reasoning=reasoning or {})\n            steps += 1\n            if obs is None:\n                print("step returned None")\n                break\n            frames.append(obs)\n            latest = obs\n            state_name = getattr(getattr(latest, "state", None), "name", str(getattr(latest, "state", None)))\n            if state_name in {"WIN", "GAME_OVER"}:\n                agent.is_done(frames, latest)\n                break\n        state_name = getattr(getattr(latest, "state", None), "name", str(getattr(latest, "state", None))) if latest is not None else "NONE"\n        report.append({"game_id": game_id, "status": state_name, "steps": steps, "levels_completed": int(getattr(latest, "levels_completed", 0) or 0) if latest else 0})\n        print("finished", report[-1])\n    final_scorecard = arc.close_scorecard(scorecard_id=scorecard_id)\n    return scorecard_id, final_scorecard, report\n'
agent_path = AGENT_DIR / "my_agent.py"
agent_path.write_text(AGENT_SOURCE)
print("wrote", agent_path)
print("bytes", agent_path.stat().st_size)


## 5. Offline syntax smoke check

This catches notebook/export mistakes before the scorecard runner is called.


In [ ]:
import py_compile
try:
    py_compile.compile(str(AGENT_DIR / "my_agent.py"), doraise=True)
    print("py_compile: OK")
except Exception as e:
    print("py_compile: FAILED", repr(e))
    raise


## 6. Optional official scorecard run

Set `NINE_ARC_RUN_SCORECARD=0` to skip this cell. In competition reruns, the package/session must be available in the environment.


In [ ]:
if not CONFIG["run_scorecard"]:
    print("Scorecard run skipped because NINE_ARC_RUN_SCORECARD=0")
elif not ARC_OK:
    print("Scorecard run skipped because arc_agi/arcengine import failed:", repr(ARC_IMPORT_ERROR))
else:
    sys.path.insert(0, str(AGENT_DIR.parent))
    sys.path.insert(0, str(AGENT_DIR))
    import importlib.util
    spec = importlib.util.spec_from_file_location("my_agent", AGENT_DIR / "my_agent.py")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    scorecard_id, final_scorecard, run_report = module.run_scorecard_with_agent(
        max_steps_per_env=CONFIG["max_steps_per_env"],
        games=CONFIG["games"],
        render_mode=CONFIG["render_mode"],
        source_url=CONFIG["source_url"],
    )
    score_report = {
        "scorecard_id": scorecard_id,
        "final_scorecard": str(final_scorecard),
        "run_report": run_report,
    }
    (REPORT_DIR / "latest_scorecard_report.json").write_text(json.dumps(score_report, indent=2, default=str))
    print(json.dumps(score_report, indent=2, default=str)[:4000])


## 7. Output files

Key artifacts created in `/kaggle/working`:

```text
agent/my_agent.py
data/skills/skill_library.json
data/reports/input_inventory.json
data/reports/component_rankings.json
data/reports/component_skill_manifest.json
data/reports/dataset_summaries.json
data/replay/transitions.jsonl
```
